# Graph Similarity Analysis Notebook

This notebook compares:

1. A **semantic embeddings graph matrix** (distance or similarity), and
2. A **subscriptions graph matrix** (distance or similarity).

It produces:

- **Numerical outputs** (correlation/error/overlap/permutation-test metrics), and
- **Visual outputs** (heatmaps, difference map, edge-weight scatter, distribution overlays, and permutation null plots).

The approach combines multiple perspectives, since no single graph metric captures all structural dimensions.


**Note:** The main analysis compares native row-normalized directed distance matrices. Any similarity transformation is visualization-only unless explicitly enabled.


## Scientific grounding

This notebook operationalizes methods inspired by:

- Mantel, N. (1967), *Cancer Research* — matrix correlation with permutation testing. https://doi.org/10.1158/0008-5472.CAN-27-2-Part-1-209
- Krackhardt, D. (1987), *Social Networks* — QAP-style permutation logic for network association.
- Borgatti, Everett, & Johnson (2018), *Analyzing Social Networks* — interpretation of dyadic matrix comparisons.
- Schieber et al. (2017), *Nature Communications* — motivation for multi-metric network dissimilarity assessment. https://doi.org/10.1038/s41467-017-01825-7

Visualization choices are aligned with standard exploratory data analysis for pairwise matrix comparisons (heatmaps for structure, scatter/distribution views for dyadic relationships, and null-distribution plots for permutation-based inference).


In [23]:
# Core scientific stack for matrix math, statistics, and plotting.
import json
import re
import shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr, spearmanr


In [24]:
# ----------------------------
# Reusable analysis primitives
# ----------------------------

@dataclass
class SimilarityResult:
    """Container for all numeric outputs from the comparison pipeline."""

    n_nodes: int
    n_pairs: int
    pearson_r: float
    pearson_p: float
    spearman_rho: float
    spearman_p: float
    cosine_similarity: float
    rmse: float
    mae: float
    edge_jaccard_sweeps: Dict[str, float]
    mantel_r: float
    mantel_p: float
    qap_r: float
    qap_p: float


def resolve_latest_drive_csv(source: Path, download_dir: Path, label: Optional[str] = None) -> Path:
    """Copy the latest Google Drive CSV artifact to local runtime storage.

    `source` can be either:
    - a direct CSV file path,
    - a directory (the notebook will read `<dir>/adjacency_matrix.csv`), or
    - a glob-style pattern so the latest modified match is selected.
    """
    source = Path(str(source))
    source_str = str(source)

    if source.exists() and source.is_dir():
        candidate = source / 'adjacency_matrix.csv'
        if not candidate.exists():
            raise FileNotFoundError(f"Expected adjacency_matrix.csv under: {source}")
        source = candidate
        source_str = str(source)

    if any(ch in source_str for ch in "*?[]"):
        matches = sorted(Path('/').glob(source_str.lstrip('/')), key=lambda p: p.stat().st_mtime)
        if not matches:
            raise FileNotFoundError(f"No Google Drive CSV matched pattern: {source_str}")
        latest_source = matches[-1]
    else:
        if not source.exists():
            raise FileNotFoundError(f"Google Drive file not found: {source}")
        latest_source = source

    download_dir.mkdir(parents=True, exist_ok=True)
    if label:
        safe_label = re.sub(r"[^A-Za-z0-9._-]+", "_", label).strip("_") or "artifact"
        destination_name = f"{safe_label}__{latest_source.name}"
    else:
        destination_name = latest_source.name
    destination = download_dir / destination_name
    shutil.copy2(latest_source, destination)
    print(f"Downloaded latest Drive artifact: {latest_source} -> {destination}")
    return destination


def load_square_matrix(csv_path: Path) -> pd.DataFrame:
    """Load a labeled square matrix from CSV and enforce validity checks.

    The CSV is expected to have a row index in the first column and matching
    row/column labels describing the same node set.
    """
    df = pd.read_csv(csv_path, index_col=0)

    if df.shape[0] != df.shape[1]:
        if df.shape[1] <= 3:
            raise ValueError(
                f"Matrix at {csv_path} is not square: {df.shape}. "
                "This looks like an edge-list export (deprecated). "
                "Use Graphiko schema adjacency matrix paths, e.g. "
                "/content/drive/MyDrive/Graphiko/graphs/embeddings_distance/latest/adjacency_matrix.csv"
            )
        raise ValueError(f"Matrix at {csv_path} is not square: {df.shape}")

    df.index = df.index.map(str)
    df.columns = df.columns.map(str)

    row_labels = set(df.index)
    col_labels = set(df.columns)
    if row_labels != col_labels:
        raise ValueError(
            f"Row/column label mismatch in {csv_path}. "
            f"Rows={len(row_labels)} unique, Cols={len(col_labels)} unique"
        )

    # Guarantee row/column alignment by ID before any downstream analysis.
    df = df.reindex(index=sorted(df.index), columns=sorted(df.columns))
    df = df.loc[df.index, df.index]
    df = df.apply(pd.to_numeric, errors="raise")
    return df


def align_matrices(a: pd.DataFrame, b: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Restrict both matrices to the shared node intersection in a stable order."""
    common = sorted(set(a.index).intersection(b.index))
    if len(common) < 3:
        raise ValueError("Need at least 3 shared nodes for robust graph similarity stats.")
    return a.loc[common, common], b.loc[common, common]


def minmax_normalize(df: pd.DataFrame) -> pd.DataFrame:
    """Global min-max normalization to [0, 1] for visualization.
    (Note: Main analysis uses native space instead).
    """
    arr = df.to_numpy(dtype=float)
    v_min, v_max = np.nanmin(arr), np.nanmax(arr)
    if np.isclose(v_min, v_max):
        return pd.DataFrame(np.zeros_like(arr), index=df.index, columns=df.columns)
    return pd.DataFrame((arr - v_min) / (v_max - v_min), index=df.index, columns=df.columns)


def distances_to_similarity(df: pd.DataFrame) -> pd.DataFrame:
    """Convert distances to similarities using similarity = 1 - normalized_distance.
    (Used only for visualization unless otherwise specified).
    """
    d = minmax_normalize(df)
    return 1.0 - d


def is_symmetric(df: pd.DataFrame, atol: float = 1e-10) -> bool:
    """Check if matrix is symmetric."""
    arr = df.to_numpy(dtype=float)
    return bool(np.allclose(arr, arr.T, atol=atol, equal_nan=True))


def max_asymmetry(df: pd.DataFrame) -> float:
    """Calculate the maximum absolute difference between a matrix and its transpose."""
    arr = df.to_numpy(dtype=float)
    return float(np.nanmax(np.abs(arr - arr.T)))


def off_diagonal_vector(df: pd.DataFrame) -> np.ndarray:
    """Vectorize matrix off-diagonal elements for directed graphs."""
    arr = df.to_numpy(dtype=float)
    # Get mask of non-diagonal elements
    mask = ~np.eye(arr.shape[0], dtype=bool)
    return arr[mask]


def permutation_distribution(
    a: pd.DataFrame,
    b: pd.DataFrame,
    permutations: int = 2000,
    random_state: int = 42,
) -> Tuple[float, np.ndarray]:
    """Observed Pearson matrix-correlation and permutation null distribution for directed graphs."""
    rng = np.random.default_rng(random_state)
    a_vec, b_vec = off_diagonal_vector(a), off_diagonal_vector(b)
    obs_r, _ = pearsonr(a_vec, b_vec)

    n = a.shape[0]
    b_arr = b.to_numpy(dtype=float)
    perm_stats = np.empty(permutations, dtype=float)

    for i in range(permutations):
        p = rng.permutation(n)
        b_perm = b_arr[np.ix_(p, p)]
        perm_stats[i], _ = pearsonr(a_vec, off_diagonal_vector(pd.DataFrame(b_perm)))

    return float(obs_r), perm_stats


def mantel_test(a: pd.DataFrame, b: pd.DataFrame, permutations: int = 2000, random_state: int = 42) -> Tuple[float, float]:
    """Mantel correlation with two-sided permutation p-value."""
    obs_r, perm_stats = permutation_distribution(a, b, permutations=permutations, random_state=random_state)
    p_value = (np.sum(np.abs(perm_stats) >= np.abs(obs_r)) + 1) / (permutations + 1)
    return float(obs_r), float(p_value)


def qap_correlation(a: pd.DataFrame, b: pd.DataFrame, permutations: int = 2000, random_state: int = 7) -> Tuple[float, float]:
    """QAP-style matrix correlation with one-sided permutation p-value."""
    obs_r, perm_stats = permutation_distribution(a, b, permutations=permutations, random_state=random_state)
    p_value = (np.sum(perm_stats >= obs_r) + 1) / (permutations + 1)
    return float(obs_r), float(p_value)


def compute_similarity_metrics(
    embeddings_matrix: pd.DataFrame,
    subscriptions_matrix: pd.DataFrame,
    edge_thresholds: List[float],
    permutations: int,
) -> SimilarityResult:
    """Compute a suite of complementary graph similarity metrics on directed graphs."""
    emb_vec = off_diagonal_vector(embeddings_matrix)
    sub_vec = off_diagonal_vector(subscriptions_matrix)

    pear_r, pear_p = pearsonr(emb_vec, sub_vec)
    spr_rho, spr_p = spearmanr(emb_vec, sub_vec)
    cos_sim = 1.0 - cosine(emb_vec, sub_vec)

    rmse = float(np.sqrt(np.mean((emb_vec - sub_vec) ** 2)))
    mae = float(np.mean(np.abs(emb_vec - sub_vec)))

    edge_jaccards = {}
    for th in edge_thresholds:
        emb_edge = emb_vec >= th
        sub_edge = sub_vec >= th
        inter = np.logical_and(emb_edge, sub_edge).sum()
        union = np.logical_or(emb_edge, sub_edge).sum()
        edge_jaccards[f"threshold_{th}"] = float(inter / union) if union > 0 else float("nan")

    mantel_r, mantel_p = mantel_test(embeddings_matrix, subscriptions_matrix, permutations)
    qap_r, qap_p = qap_correlation(embeddings_matrix, subscriptions_matrix, permutations)

    return SimilarityResult(
        n_nodes=embeddings_matrix.shape[0],
        n_pairs=emb_vec.size,
        pearson_r=float(pear_r),
        pearson_p=float(pear_p),
        spearman_rho=float(spr_rho),
        spearman_p=float(spr_p),
        cosine_similarity=float(cos_sim),
        rmse=rmse,
        mae=mae,
        edge_jaccard_sweeps=edge_jaccards,
        mantel_r=mantel_r,
        mantel_p=mantel_p,
        qap_r=qap_r,
        qap_p=qap_p,
    )


def save_visualizations(
    embeddings_matrix: pd.DataFrame,
    subscriptions_matrix: pd.DataFrame,
    output_dir: Path,
    embeddings_analysis: pd.DataFrame = None,
    subscriptions_analysis: pd.DataFrame = None,
    permutations: int = 2000,
) -> Dict[str, str]:
    """Render and save graph comparison figures."""
    output_dir.mkdir(parents=True, exist_ok=True)
    out: Dict[str, str] = {}

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    sns.heatmap(embeddings_matrix, cmap="viridis", ax=axes[0], cbar=True)
    axes[0].set_title("Embeddings Graph (directed)")
    sns.heatmap(subscriptions_matrix, cmap="viridis", ax=axes[1], cbar=True)
    axes[1].set_title("Subscriptions Graph (directed)")
    p1 = output_dir / "heatmaps_side_by_side.png"
    fig.savefig(p1, dpi=220)
    plt.close(fig)
    out["heatmaps"] = str(p1)

    diff = embeddings_matrix - subscriptions_matrix
    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    sns.heatmap(diff, cmap="coolwarm", center=0.0, ax=ax)
    ax.set_title("Difference Matrix (Embeddings - Subscriptions)")
    p2 = output_dir / "difference_heatmap.png"
    fig.savefig(p2, dpi=220)
    plt.close(fig)
    out["difference_heatmap"] = str(p2)

    emb_vec = off_diagonal_vector(embeddings_matrix)
    sub_vec = off_diagonal_vector(subscriptions_matrix)
    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    sns.regplot(x=emb_vec, y=sub_vec, scatter_kws={"alpha": 0.45, "s": 18}, ax=ax)
    ax.set_xlabel("Embeddings directed edge weight")
    ax.set_ylabel("Subscriptions directed edge weight")
    ax.set_title("Ordered pair relationship between graphs")
    p3 = output_dir / "edge_weight_scatter.png"
    fig.savefig(p3, dpi=220)
    plt.close(fig)
    out["scatter"] = str(p3)

    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    sns.histplot(emb_vec, bins=40, color="#1f77b4", alpha=0.45, stat="density", label="Embeddings", ax=ax)
    sns.histplot(sub_vec, bins=40, color="#ff7f0e", alpha=0.45, stat="density", label="Subscriptions", ax=ax)
    ax.set_xlabel("Directed edge weight")
    ax.set_ylabel("Density")
    ax.set_title("Directed edge-weight distribution overlap")
    ax.legend()
    p4 = output_dir / "edge_weight_distributions.png"
    fig.savefig(p4, dpi=220)
    plt.close(fig)
    out["distribution_overlap"] = str(p4)

    mantel_obs, mantel_perm = permutation_distribution(embeddings_analysis if embeddings_analysis is not None else embeddings_matrix, subscriptions_analysis if subscriptions_analysis is not None else subscriptions_matrix, permutations=permutations, random_state=42)
    qap_obs, qap_perm = permutation_distribution(embeddings_analysis if embeddings_analysis is not None else embeddings_matrix, subscriptions_analysis if subscriptions_analysis is not None else subscriptions_matrix, permutations=permutations, random_state=7)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    sns.histplot(mantel_perm, bins=40, stat="density", color="#2ca02c", alpha=0.75, ax=axes[0])
    axes[0].axvline(mantel_obs, color="black", linestyle="--", linewidth=2, label=f"Observed r={mantel_obs:.3f}")
    axes[0].set_title("Mantel permutation null distribution")
    axes[0].set_xlabel("Permutation correlation")
    axes[0].legend()

    sns.histplot(qap_perm, bins=40, stat="density", color="#9467bd", alpha=0.75, ax=axes[1])
    axes[1].axvline(qap_obs, color="black", linestyle="--", linewidth=2, label=f"Observed r={qap_obs:.3f}")
    axes[1].set_title("QAP permutation null distribution")
    axes[1].set_xlabel("Permutation correlation")
    axes[1].legend()

    p5 = output_dir / "permutation_null_distributions.png"
    fig.savefig(p5, dpi=220)
    plt.close(fig)
    out["permutation_nulls"] = str(p5)

    return out



## Configure your run

Update the two Google Drive input paths (or glob patterns) below.

- The notebook copies the **latest Google Drive artifact** for each input into local runtime storage before analysis.
- If your files represent **distance matrices**, keep `DISTANCE_TO_SIMILARITY = True`.
- If they are already **similarities**, set it to `False`.


In [25]:
# ----------------------------
# User configuration
# ----------------------------
EMBEDDINGS_DRIVE_SOURCE = Path('/content/drive/MyDrive/Graphiko/graphs/embeddings_distance/latest/adjacency_matrix.csv')
SUBSCRIPTIONS_DRIVE_SOURCE = Path('/content/drive/MyDrive/Graphiko/graphs/subscriptions_normalized_distance/latest/adjacency_matrix.csv')
LOCAL_DOWNLOAD_DIR = Path('inputs/downloaded_from_drive')
OUTPUT_DIR = Path('outputs/graph_similarity')

ANALYSIS_SPACE = "native_distance"
VISUALIZATION_SPACE = "distance" # or "similarity"
COMPARISON_MODE = "directed_row_normalized_distance"
EDGE_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]
PERMUTATIONS = 2000



In [26]:
# ----------------------------
# Execute analysis
# ----------------------------
EMBEDDINGS_CSV = resolve_latest_drive_csv(EMBEDDINGS_DRIVE_SOURCE, LOCAL_DOWNLOAD_DIR, label='embeddings')
SUBSCRIPTIONS_CSV = resolve_latest_drive_csv(SUBSCRIPTIONS_DRIVE_SOURCE, LOCAL_DOWNLOAD_DIR, label='subscriptions')

emb_raw = load_square_matrix(EMBEDDINGS_CSV)
sub_raw = load_square_matrix(SUBSCRIPTIONS_CSV)

emb_aligned, sub_aligned = align_matrices(emb_raw, sub_raw)

# Calculate symmetry diagnostics before analysis
emb_is_sym = is_symmetric(emb_aligned)
sub_is_sym = is_symmetric(sub_aligned)
emb_max_asym = max_asymmetry(emb_aligned)
sub_max_asym = max_asymmetry(sub_aligned)

# Main analysis happens in ANALYSIS_SPACE (native distance by default)
if ANALYSIS_SPACE == "native_distance":
    emb_analysis = emb_aligned
    sub_analysis = sub_aligned
else:
    # If explicitly requested to use something else for main analysis
    emb_analysis = minmax_normalize(emb_aligned)
    sub_analysis = minmax_normalize(sub_aligned)
    if ANALYSIS_SPACE == "similarity":
        emb_analysis = distances_to_similarity(emb_aligned)
        sub_analysis = distances_to_similarity(sub_aligned)

result = compute_similarity_metrics(
    emb_analysis,
    sub_analysis,
    edge_thresholds=EDGE_THRESHOLDS,
    permutations=PERMUTATIONS,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics = asdict(result)

metrics_json = OUTPUT_DIR / 'similarity_metrics.json'
metrics_csv = OUTPUT_DIR / 'similarity_metrics.csv'
summary_json = OUTPUT_DIR / 'run_summary.json'

with metrics_json.open('w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

# Flatten edge_jaccard_sweeps for CSV
metrics_flat = metrics.copy()
jaccards = metrics_flat.pop('edge_jaccard_sweeps')
for k, v in jaccards.items():
    metrics_flat[f"edge_jaccard_{k}"] = v

pd.DataFrame([metrics_flat]).to_csv(metrics_csv, index=False)

# Visualization
if VISUALIZATION_SPACE == "similarity":
    emb_viz = distances_to_similarity(emb_aligned)
    sub_viz = distances_to_similarity(sub_aligned)
elif VISUALIZATION_SPACE == "normalized_distance":
    emb_viz = minmax_normalize(emb_aligned)
    sub_viz = minmax_normalize(sub_aligned)
else: # native distance
    emb_viz = emb_aligned
    sub_viz = sub_aligned

plots = save_visualizations(emb_viz, sub_viz, OUTPUT_DIR, embeddings_analysis=emb_analysis, subscriptions_analysis=sub_analysis, permutations=PERMUTATIONS)

summary = {
    'inputs': {
        'embeddings_drive_source': str(EMBEDDINGS_DRIVE_SOURCE),
        'subscriptions_drive_source': str(SUBSCRIPTIONS_DRIVE_SOURCE),
        'embeddings_downloaded_csv': str(EMBEDDINGS_CSV),
        'subscriptions_downloaded_csv': str(SUBSCRIPTIONS_CSV),
        'analysis_space': ANALYSIS_SPACE,
        'visualization_space': VISUALIZATION_SPACE,
        'comparison_mode': COMPARISON_MODE,
    },
    'diagnostics': {
        'embeddings_is_symmetric': emb_is_sym,
        'embeddings_max_asymmetry': emb_max_asym,
        'subscriptions_is_symmetric': sub_is_sym,
        'subscriptions_max_asymmetry': sub_max_asym,
    },
    'metrics_json': str(metrics_json),
    'metrics_csv': str(metrics_csv),
    'plots': plots,
}

with summary_json.open('w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Graph similarity analysis complete.')
print(json.dumps(summary, indent=2))
pd.DataFrame([metrics_flat]).T.rename(columns={0: 'value'})

